# Perbandingan performa Naive Bayes dan SVM dalam Deteksi Teks Parafrase AI pada Jurnal Ilmiah Berbahasa Indonesia


Struktur notebook ini:
Import Modul → Pemuatan Data → Preprocessing (per tahap) → Pembagian Data → Pembobotan TF-IDF →
Penerapan Model → Evaluasi Model → Confusion Matrix → Visualisasi Data. Dua penyesuaian disengaja
tidak ada SMOTE (dataset sudah hampir seimbang), dan tidak ada stopword removal/stemming (dibutuhkan
justru variasi kata fungsi dan imbuhan sebagai fitur pembeda, bukan makna semantik).

## Import Modul

In [24]:

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from scipy.stats import binomtest

import matplotlib.pyplot as plt
import seaborn as sns


## Pemuatan Data

In [25]:

# Membaca dataset mentah
df_raw = pd.read_csv('dataset_parafrase_berlabel.csv')
df_raw.columns = ['teks', 'label']
df_raw = df_raw.dropna(subset=['teks', 'label'])

print(f"Total data: {df_raw.shape[0]}")
df_raw.tail(5)


Total data: 604


,teks,label
599,Emosi masyarakat sering tersampaikan melalui k...,1
600,Hasil penelitian analisis sentimen mencakup im...,1
601,Analisis sentimen merupakan pendekatan dalam i...,1
602,Analisis sentimen berfungsi sebagai instrumen ...,1
603,Twitter telah menjadi platform yang signifikan...,1


In [26]:

# Distribusi label awal
print("Distribusi Label Awal:")
print(df_raw['label'].value_counts().rename({0: 'Manusia', 1: 'AI'}))


Distribusi Label Awal:
label
Manusia    304
AI         300
Name: count, dtype: int64


In [27]:

# Rekonstruksi id_topik: tiap paragraf AI dipasangkan ke paragraf manusia dengan
# kemiripan TF-IDF (cosine similarity) tertinggi, karena berkas mentah tidak
# menyimpan penanda pasangan topik secara eksplisit. id_topik dipakai pada
# tahap Pembagian Data agar satu topik tidak terpecah antara data latih & uji.
df_manusia = df_raw[df_raw['label'] == 0].copy().reset_index(drop=True)
df_ai      = df_raw[df_raw['label'] == 1].copy().reset_index(drop=True)
df_manusia['id_topik'] = range(1, len(df_manusia) + 1)

_tfidf_pairing = TfidfVectorizer()
_tf_h = _tfidf_pairing.fit_transform(df_manusia['teks'])
_tf_a = _tfidf_pairing.transform(df_ai['teks'])
_sim  = cosine_similarity(_tf_a, _tf_h)
df_ai['id_topik'] = [df_manusia.loc[_sim[i].argmax(), 'id_topik'] for i in range(len(df_ai))]

df = pd.concat([df_manusia, df_ai]).sort_values(['id_topik', 'label']).reset_index(drop=True)
print(f"Jumlah id_topik unik: {df['id_topik'].nunique()}")
df.tail(5)


Jumlah id_topik unik: 304


,teks,label,id_topik
599,Twitter adalah platform media sosial yang ...,0,301
600,Twitter telah menjadi platform yang signifikan...,1,301
601,Klasifikasi adalah landasan dasar dalam ana...,0,302
602,Naïve Bayes Classifier adalah algoritma machin...,0,303
603,Analisis sentiment bagian dari penambahan teks...,0,304


## PREPROCESSING

### Case Folding

In [28]:

# Mengubah semua huruf menjadi kecil (lowercase)
df['teks_case_folding'] = df['teks'].str.lower()

print("Data Setelah Case Folding:")
df.tail(5)


Data Setelah Case Folding:


,teks,label,id_topik,teks_case_folding
599,Twitter adalah platform media sosial yang ...,0,301,twitter adalah platform media sosial yang ...
600,Twitter telah menjadi platform yang signifikan...,1,301,twitter telah menjadi platform yang signifikan...
601,Klasifikasi adalah landasan dasar dalam ana...,0,302,klasifikasi adalah landasan dasar dalam ana...
602,Naïve Bayes Classifier adalah algoritma machin...,0,303,naïve bayes classifier adalah algoritma machin...
603,Analisis sentiment bagian dari penambahan teks...,0,304,analisis sentiment bagian dari penambahan teks...


### Cleansing URL

In [29]:

# Menghapus tautan (URL) berawalan http, https, atau www
df['teks_no_url'] = df['teks_case_folding'].apply(
    lambda t: re.sub(r'http\S+|www\S+|https\S+', '', t, flags=re.MULTILINE)
)

print("Data Setelah Cleansing URL:")
df.tail(5)


Data Setelah Cleansing URL:


,teks,label,id_topik,teks_case_folding,teks_no_url
599,Twitter adalah platform media sosial yang ...,0,301,twitter adalah platform media sosial yang ...,twitter adalah platform media sosial yang ...
600,Twitter telah menjadi platform yang signifikan...,1,301,twitter telah menjadi platform yang signifikan...,twitter telah menjadi platform yang signifikan...
601,Klasifikasi adalah landasan dasar dalam ana...,0,302,klasifikasi adalah landasan dasar dalam ana...,klasifikasi adalah landasan dasar dalam ana...
602,Naïve Bayes Classifier adalah algoritma machin...,0,303,naïve bayes classifier adalah algoritma machin...,naïve bayes classifier adalah algoritma machin...
603,Analisis sentiment bagian dari penambahan teks...,0,304,analisis sentiment bagian dari penambahan teks...,analisis sentiment bagian dari penambahan teks...


### Cleansing Tanda Baca dan Angka

In [30]:

# Menyisakan hanya huruf a-z dan spasi; tanda baca, angka, simbol, dan emoji dibuang
df['teks_alfabet_saja'] = df['teks_no_url'].apply(lambda t: re.sub(r'[^a-z\s]', ' ', t))

print("Data Setelah Cleansing Tanda Baca & Angka:")
df.tail(5)


Data Setelah Cleansing Tanda Baca & Angka:


,teks,label,id_topik,teks_case_folding,teks_no_url,teks_alfabet_saja
599,Twitter adalah platform media sosial yang ...,0,301,twitter adalah platform media sosial yang ...,twitter adalah platform media sosial yang ...,twitter adalah platform media sosial yang ...
600,Twitter telah menjadi platform yang signifikan...,1,301,twitter telah menjadi platform yang signifikan...,twitter telah menjadi platform yang signifikan...,twitter telah menjadi platform yang signifikan...
601,Klasifikasi adalah landasan dasar dalam ana...,0,302,klasifikasi adalah landasan dasar dalam ana...,klasifikasi adalah landasan dasar dalam ana...,klasifikasi adalah landasan dasar dalam ana...
602,Naïve Bayes Classifier adalah algoritma machin...,0,303,naïve bayes classifier adalah algoritma machin...,naïve bayes classifier adalah algoritma machin...,na ve bayes classifier adalah algoritma machin...
603,Analisis sentiment bagian dari penambahan teks...,0,304,analisis sentiment bagian dari penambahan teks...,analisis sentiment bagian dari penambahan teks...,analisis sentiment bagian dari penambahan teks...


### Normalisasi Spasi

In [31]:

# Memadatkan spasi berulang menjadi satu spasi, membuang spasi di awal/akhir teks
df['teks_final'] = df['teks_alfabet_saja'].apply(lambda t: re.sub(r'\s+', ' ', t).strip())

print("Data Setelah Normalisasi Spasi (hasil akhir preprocessing):")
df.tail(5)


Data Setelah Normalisasi Spasi (hasil akhir preprocessing):


,teks,label,id_topik,teks_case_folding,teks_no_url,teks_alfabet_saja,teks_final
599,Twitter adalah platform media sosial yang ...,0,301,twitter adalah platform media sosial yang ...,twitter adalah platform media sosial yang ...,twitter adalah platform media sosial yang ...,twitter adalah platform media sosial yang memu...
600,Twitter telah menjadi platform yang signifikan...,1,301,twitter telah menjadi platform yang signifikan...,twitter telah menjadi platform yang signifikan...,twitter telah menjadi platform yang signifikan...,twitter telah menjadi platform yang signifikan...
601,Klasifikasi adalah landasan dasar dalam ana...,0,302,klasifikasi adalah landasan dasar dalam ana...,klasifikasi adalah landasan dasar dalam ana...,klasifikasi adalah landasan dasar dalam ana...,klasifikasi adalah landasan dasar dalam analis...
602,Naïve Bayes Classifier adalah algoritma machin...,0,303,naïve bayes classifier adalah algoritma machin...,naïve bayes classifier adalah algoritma machin...,na ve bayes classifier adalah algoritma machin...,na ve bayes classifier adalah algoritma machin...
603,Analisis sentiment bagian dari penambahan teks...,0,304,analisis sentiment bagian dari penambahan teks...,analisis sentiment bagian dari penambahan teks...,analisis sentiment bagian dari penambahan teks...,analisis sentiment bagian dari penambahan teks...


### Catatan: Stopword Removal dan Stemming Tidak Diterapkan
Pada penelitian ini tidak menerapkan stopword removal dan stemming karena penelitian ini bukan untuk menangkap makna melainkan membedakan gaya penulisan manusia vs AI, dan kata fungsi ("yang", "dan", "di") beserta variasi imbuhan adalah bagian dari gaya itu sendiri jika dihapus/dipangkas, jejak yang mau dideteksi ikut hilang. Oleh karena itu pipeline berhenti di normalisasi spasi.


In [32]:

# Simpan hasil preprocessing untuk dipakai pada tahap selanjutnya
df_bersih = df[['teks_final', 'label', 'id_topik']].dropna().reset_index(drop=True)
df_bersih = df_bersih[df_bersih['teks_final'].str.strip().astype(bool)]

print(f"Jumlah data setelah preprocessing: {df_bersih.shape[0]}")
df_bersih.tail(5)


Jumlah data setelah preprocessing: 604


,teks_final,label,id_topik
599,twitter adalah platform media sosial yang memu...,0,301
600,twitter telah menjadi platform yang signifikan...,1,301
601,klasifikasi adalah landasan dasar dalam analis...,0,302
602,na ve bayes classifier adalah algoritma machin...,0,303
603,analisis sentiment bagian dari penambahan teks...,0,304


## PEMBAGIAN DATA

Pada penelitian ini dipakai `StratifiedGroupKFold` (5 fold) sebagai ganti dari pembagian data seperti (80:20), dengan dua alasan:
*   (1) `groups=id_topik` menjaga agar satu topik (paragraf manusia + pasangan
AI-nya) tidak pernah terpecah antara data latih dan data uji
*   (2) 5 fold memberi 5 estimasi performa (rata-rata ± simpangan baku) alih-alih satu angka dari satu kali pembagian acak.



In [33]:

X = df_bersih['teks_final'].values
y = df_bersih['label'].values
groups = df_bersih['id_topik'].values

N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
splits = list(sgkf.split(X, y, groups))

for i, (train_idx, test_idx) in enumerate(splits):
    bocor = len(set(groups[train_idx]) & set(groups[test_idx]))
    print(f"Fold {i+1}: Data latih={len(train_idx)} baris | Data uji={len(test_idx)} baris | Topik bocor={bocor}")


Fold 1: Data latih=478 baris | Data uji=126 baris | Topik bocor=0
Fold 2: Data latih=487 baris | Data uji=117 baris | Topik bocor=0
Fold 3: Data latih=482 baris | Data uji=122 baris | Topik bocor=0
Fold 4: Data latih=481 baris | Data uji=123 baris | Topik bocor=0
Fold 5: Data latih=488 baris | Data uji=116 baris | Topik bocor=0


## PEMBOBOTAN KATA TF-IDF

Berbeda dari pembobotan kata lain yang memakai `TfidfVectorizer()` default (kata utuh), penelitian ini memakai fitur karakter n-gram (2-6 huruf berurutan) karena pola pembeda manusia-vs-AI diduga bermukim di tingkat sub-kata (ejaan, imbuhan), bukan pilihan kata. Vectorizer di-*fit* ulang di dalam setiap fold didemonstrasikan dulu pada Fold 1 di sini, sebelum dipakai berulang pada seluruh fold di bagian Penerapan Model.

In [34]:

tfidf_params = dict(analyzer='char', ngram_range=(2, 6), max_features=10000,
                     sublinear_tf=True, min_df=2)

# Ilustrasi pada Fold 1
train_idx, test_idx = splits[0]
tfidf_vectorizer = TfidfVectorizer(**tfidf_params)

X_train_tfidf = tfidf_vectorizer.fit_transform(X[train_idx])   # fit HANYA di data latih
X_test_tfidf  = tfidf_vectorizer.transform(X[test_idx])        # transform saja di data uji

print(f"Bentuk matriks TF-IDF data latih: {X_train_tfidf.shape}")
print(f"Bentuk matriks TF-IDF data uji  : {X_test_tfidf.shape}")
print(f"Jumlah fitur (n-gram karakter) dalam vocabulary: {len(tfidf_vectorizer.get_feature_names_out())}")


Bentuk matriks TF-IDF data latih: (478, 10000)
Bentuk matriks TF-IDF data uji  : (126, 10000)
Jumlah fitur (n-gram karakter) dalam vocabulary: 10000


In [35]:

print("Distribusi label pada seluruh data (setelah preprocessing):")
print(pd.Series(y).value_counts().rename({0: 'Manusia', 1: 'AI'}))


Distribusi label pada seluruh data (setelah preprocessing):
Manusia    304
AI         300
Name: count, dtype: int64


## HYPERPARAMETER TUNING

Kode referensi memakai `alpha=0.5` tanpa pencarian. Di sini alpha (MNB) dan C (SVM) dicari lewat `GridSearchCV` memakai skema pembagian data yang sama (`splits`), supaya nilai yang dipakai bisa dipertanggungjawabkan, bukan hanya nilai default library.

In [ ]:

pipe_nb  = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', MultinomialNB())])
pipe_svm = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', SVC(kernel='linear', random_state=42))])

grid_nb  = GridSearchCV(pipe_nb,  {'clf__alpha': [0.1, 0.5, 1.0, 2.0]}, cv=splits, scoring='f1_weighted', n_jobs=-1)
grid_svm = GridSearchCV(pipe_svm, {'clf__C':     [0.01, 0.1, 1.0, 10.0]}, cv=splits, scoring='f1_weighted', n_jobs=-1)

grid_nb.fit(X, y)
grid_svm.fit(X, y)

print(f"MNB - alpha terbaik : {grid_nb.best_params_['clf__alpha']}  (F1-weighted CV = {grid_nb.best_score_:.4f})")
print(f"SVM - C terbaik     : {grid_svm.best_params_['clf__C']}  (F1-weighted CV = {grid_svm.best_score_:.4f})")


## PENERAPAN MODEL

Alur TRAIN → TEST dijalankan pada tiap fold: TF-IDF di-*fit* pada data latih fold tersebut, model dilatih, lalu data uji ditransformasi (bukan di-*fit* ulang) dan diprediksi. Fungsi berikut mengulang alur ini untuk kelima fold, dipanggil untuk MNB dan SVM, versi default maupun tuned.

In [ ]:

def latih_dan_uji_satu_fold(model, param_name, param_value, train_idx, test_idx):
    # --- TRAIN ---
    tfidf = TfidfVectorizer(**tfidf_params)
    X_train_vec = tfidf.fit_transform(X[train_idx])
    model.set_params(**{param_name: param_value})
    model.fit(X_train_vec, y[train_idx])

    # --- TEST ---
    X_test_vec = tfidf.transform(X[test_idx])
    pred_test  = model.predict(X_test_vec)
    pred_train = model.predict(X_train_vec)  # dipakai untuk pengecekan overfitting

    return pred_test, pred_train

def evaluasi_5fold(model, param_name, param_value):
    acc_test, prec_test, rec_test, f1_test, acc_train = [], [], [], [], []
    oof_pred = np.zeros_like(y)

    for train_idx, test_idx in splits:
        pred_test, pred_train = latih_dan_uji_satu_fold(model, param_name, param_value, train_idx, test_idx)
        acc_test.append(accuracy_score(y[test_idx], pred_test))
        prec_test.append(precision_score(y[test_idx], pred_test, average='weighted'))
        rec_test.append(recall_score(y[test_idx], pred_test, average='weighted'))
        f1_test.append(f1_score(y[test_idx], pred_test, average='weighted'))
        acc_train.append(accuracy_score(y[train_idx], pred_train))
        oof_pred[test_idx] = pred_test

    return {
        'acc_test': np.array(acc_test), 'prec_test': np.array(prec_test),
        'rec_test': np.array(rec_test), 'f1_test': np.array(f1_test),
        'acc_train': np.array(acc_train), 'oof_pred': oof_pred,
    }

hasil_nb_default  = evaluasi_5fold(MultinomialNB(), 'alpha', 1.0)
hasil_nb_tuned    = evaluasi_5fold(MultinomialNB(), 'alpha', grid_nb.best_params_['clf__alpha'])
hasil_svm_default = evaluasi_5fold(SVC(kernel='linear', random_state=42), 'C', 1.0)
hasil_svm_tuned   = evaluasi_5fold(SVC(kernel='linear', random_state=42), 'C', grid_svm.best_params_['clf__C'])

print("Pelatihan & pengujian 5-fold selesai untuk MNB (default & tuned) dan SVM (default & tuned).")


## EVALUASI MODEL

In [ ]:

def ringkas(nama, h):
    print(f"{nama:28s}  Akurasi={h['acc_test'].mean()*100:.2f}%±{h['acc_test'].std()*100:.2f}  "
          f"Precision={h['prec_test'].mean()*100:.2f}%  Recall={h['rec_test'].mean()*100:.2f}%  "
          f"F1={h['f1_test'].mean()*100:.2f}%±{h['f1_test'].std()*100:.2f}")

print("Hasil evaluasi 5-fold (rata-rata ± simpangan baku pada data uji):")
print("-" * 100)
ringkas("MNB  (alpha=1.0, default)", hasil_nb_default)
ringkas(f"MNB  (alpha={grid_nb.best_params_['clf__alpha']}, tuned)", hasil_nb_tuned)
ringkas("SVM  (C=1.0, default)", hasil_svm_default)
ringkas(f"SVM  (C={grid_svm.best_params_['clf__C']}, tuned)", hasil_svm_tuned)


In [ ]:

print("\nLaporan Klasifikasi - MNB (tuned, prediksi out-of-fold):")
print(classification_report(y, hasil_nb_tuned['oof_pred'], target_names=['Manusia', 'AI']))

print("\nLaporan Klasifikasi - SVM (tuned, prediksi out-of-fold):")
print(classification_report(y, hasil_svm_tuned['oof_pred'], target_names=['Manusia', 'AI']))


### Pemeriksaan Overfitting (Akurasi Latih vs Uji)

In [ ]:

gap_table = pd.DataFrame({
    'Fold': range(1, N_SPLITS + 1),
    'MNB_Train': hasil_nb_tuned['acc_train'], 'MNB_Test': hasil_nb_tuned['acc_test'],
    'MNB_Gap':   hasil_nb_tuned['acc_train'] - hasil_nb_tuned['acc_test'],
    'SVM_Train': hasil_svm_tuned['acc_train'], 'SVM_Test': hasil_svm_tuned['acc_test'],
    'SVM_Gap':   hasil_svm_tuned['acc_train'] - hasil_svm_tuned['acc_test'],
})
display(gap_table.round(4))
print(f"Rata-rata gap MNB: {gap_table['MNB_Gap'].mean():.4f}  |  Rata-rata gap SVM: {gap_table['SVM_Gap'].mean():.4f}")


## CONFUSION MATRIX

In [ ]:

cm_nb  = confusion_matrix(y, hasil_nb_tuned['oof_pred'])
cm_svm = confusion_matrix(y, hasil_svm_tuned['oof_pred'])
kelas = ['Manusia', 'AI']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', xticklabels=kelas, yticklabels=kelas, ax=axes[0])
axes[0].set_xlabel('Prediksi Model'); axes[0].set_ylabel('Label Aktual')
axes[0].set_title(f"Confusion Matrix - MNB (alpha={grid_nb.best_params_['clf__alpha']})")

sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges', xticklabels=kelas, yticklabels=kelas, ax=axes[1])
axes[1].set_xlabel('Prediksi Model'); axes[1].set_ylabel('Label Aktual')
axes[1].set_title(f"Confusion Matrix - SVM (C={grid_svm.best_params_['clf__C']})")

plt.tight_layout()
plt.savefig('confusion_matrix_oof.png', bbox_inches='tight', dpi=150)
plt.show()


## UJI SIGNIFIKANSI (MCNEMAR)

Penelitian ini membandingkan dua algoritma pada instance uji yang identik, sehingga uji McNemar relevan untuk menjawab apakah selisih performa MNB vs SVM konsisten atau sekadar variasi sampling.

In [ ]:

benar_nb  = (hasil_nb_tuned['oof_pred'] == y)
benar_svm = (hasil_svm_tuned['oof_pred'] == y)

b = int(np.sum(benar_nb & ~benar_svm))   # NB benar, SVM salah
c = int(np.sum(~benar_nb & benar_svm))   # NB salah, SVM benar
n_diskordan = b + c

print(f"NB benar & SVM salah (b): {b}  |  NB salah & SVM benar (c): {c}  |  Total diskordan: {n_diskordan}")

if n_diskordan > 0:
    p_value = binomtest(min(b, c), n_diskordan, 0.5, alternative='two-sided').pvalue
    print(f"McNemar (uji binomial eksak): p-value = {p_value:.4f}")
    if p_value < 0.05:
        print(f"Signifikan (alpha=0,05): {'MNB' if b > c else 'SVM'} unggul secara statistik.")
    else:
        print("Tidak signifikan (alpha=0,05): selisih MNB vs SVM tidak cukup kuat untuk")
        print("disimpulkan sebagai perbedaan yang konsisten.")


## VISUALISASI DATA

In [ ]:

# Distribusi persentase label (mengikuti gaya pie chart pada kode referensi)
label_counts = pd.Series(y).value_counts().rename({0: 'Manusia', 1: 'AI'})
colors = ['#4682B4', '#FF6347']

plt.figure(figsize=(7, 7))
plt.pie(label_counts, explode=(0, 0.08), labels=label_counts.index, colors=colors,
        autopct='%1.1f%%', shadow=True, startangle=140)
plt.title('Distribusi Persentase Data Berdasarkan Label', fontsize=14)
plt.axis('equal')
plt.show()


In [ ]:

# Perbandingan F1-Score keempat skenario model
fig, ax = plt.subplots(figsize=(9, 5))
skenario = ['MNB\n(default)', 'MNB\n(tuned)', 'SVM\n(default)', 'SVM\n(tuned)']
f1_rerata = [hasil_nb_default['f1_test'].mean(), hasil_nb_tuned['f1_test'].mean(),
             hasil_svm_default['f1_test'].mean(), hasil_svm_tuned['f1_test'].mean()]
f1_std    = [hasil_nb_default['f1_test'].std(), hasil_nb_tuned['f1_test'].std(),
             hasil_svm_default['f1_test'].std(), hasil_svm_tuned['f1_test'].std()]

bars = ax.bar(skenario, f1_rerata, yerr=f1_std, capsize=6, color=['#4682B4', '#1f4e79', '#FF6347', '#a83214'])
ax.set_ylabel('F1-Score (weighted, rata-rata 5-fold)')
ax.set_ylim(0, 1)
ax.set_title('Perbandingan F1-Score Antarmodel (± simpangan baku)')
for bar, val in zip(bars, f1_rerata):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val*100:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('perbandingan_f1_model.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:

# Top n-gram karakter paling dominan tiap kelas (model final, dilatih pada seluruh data)
tfidf_final = TfidfVectorizer(**tfidf_params)
X_final = tfidf_final.fit_transform(X)
vocab_final = tfidf_final.get_feature_names_out()
model_nb_final = MultinomialNB(alpha=grid_nb.best_params_['clf__alpha']).fit(X_final, y)

top_n = 15
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for i, (kls, warna) in enumerate(zip(['Manusia', 'AI'], ['#4682B4', '#FF6347'])):
    log_prob = model_nb_final.feature_log_prob_[i]
    top_idx = np.argsort(log_prob)[-top_n:][::-1]
    axes[i].barh([vocab_final[j] for j in top_idx][::-1], log_prob[top_idx][::-1], color=warna)
    axes[i].set_title(f'Top {top_n} Fitur N-gram - Kelas {kls}')
    axes[i].set_xlabel('Log Probability')
plt.tight_layout()
plt.savefig('fitur_naive_bayes_final.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nCatatan: model final ini dilatih pada seluruh data untuk keperluan visualisasi fitur,")
print("bukan untuk estimasi performa. Estimasi performa yang sah merujuk pada bagian Evaluasi Model di atas.")
